# PySpark Fundamentals — Part 10: Spark UI, Spark Internals, and Access in Local, Google Colab, and Databricks

> **Learning goal:** Understand Spark fundamentals, follow code from submission to task execution, and learn how to access and interpret Spark UI in local development, Google Colab, and Databricks.
>
> **Version note:** UI labels and platform navigation vary by Spark version, deployment mode, and workspace. The concepts below are general; managed platforms may customize the interface.

## Table of Contents
1. Why Spark UI matters
2. Spark fundamentals
3. Architecture and components
4. What happens when code runs
5. Lazy evaluation, transformations, and actions
6. Jobs, stages, tasks, partitions, and DAGs
7. Spark UI in local mode
8. Spark UI in Google Colab
9. Spark UI in Databricks
10. Spark UI tabs
11. Reading a job end to end
12. Query plans and Spark SQL internals
13. Performance diagnosis
14. Event logs and History Server
15. Useful configuration properties
16. Practical investigation labs
17. Common mistakes
18. Assignments
19. Glossary

---

# 1. Why Spark UI Matters

The **Spark UI** is a web interface for inspecting a Spark application's execution. While an application is running, the live UI shows jobs, stages, tasks, executors, SQL plans, storage use, and runtime metrics. A History Server or a managed-platform history view may show completed applications.

It helps answer:
- Which jobs did my actions trigger?
- Which stages and tasks took the most time?
- How many partitions were processed?
- Was there a large shuffle, disk spill, high garbage collection time, or uneven task duration?
- Which DataFrame/SQL operator introduced an exchange or join strategy?
- Did tasks or executors fail?
- Is cached data consuming memory?

**Mental model:** The UI gives evidence about execution. It does not automatically identify the root cause or optimize the code for you.

# 2. Spark Fundamentals

## 2.1 What is Apache Spark?

Apache Spark is a distributed compute engine for batch ETL, SQL analytics, streaming, machine learning, and other data-processing workloads. Spark is not itself a storage system: it can read and write data in filesystems, object stores, databases, and lakehouse formats.

Spark can run on one computer in local mode, on a cluster, or through managed platforms such as Databricks.

## 2.2 Spark vs Hadoop components

- **HDFS** is a distributed filesystem.
- **YARN** is a cluster resource manager commonly used in Hadoop environments.
- **Spark** is a compute engine.
- Spark can run without HDFS and can use different deployment/resource-management options.

## 2.3 Core architecture

```text
User code / notebook
        |
        v
      Driver
  - SparkSession / SparkContext
  - Plans and schedules work
  - Tracks jobs, stages, tasks
        |
        +--------------------------+
        |                          |
        v                          v
    Executor A                 Executor B
    - Runs tasks               - Runs tasks
    - Caches partitions        - Caches partitions
    - Reports metrics          - Reports metrics
        |                          |
        v                          v
    Data partitions             Data partitions

Cluster manager allocates resources in cluster deployments.
```

In local mode, task execution uses local threads on the same machine; there may not be separate remote executor machines. In cluster mode, the driver/executor placement depends on deployment mode and cluster manager.

## 2.4 Components

| Component | Responsibility |
|---|---|
| Driver | Coordinates the application, builds plans, schedules work, tracks results and metrics |
| SparkSession | Main entry point for DataFrame and Spark SQL work |
| SparkContext | Core application context; usually created through SparkSession |
| Executor | Process that runs tasks and can cache data |
| Cluster manager | Allocates resources in cluster deployments, e.g. Kubernetes, YARN, standalone Spark |
| Job | Execution work triggered by an action/request |
| Stage | Group of tasks separated from dependent work, often at shuffle boundaries |
| Task | Scheduled unit of work, typically processing one partition in a stage |
| Partition | Logical chunk of a dataset that can be processed in parallel |
| DAG | Directed acyclic graph of dependencies between operations |
| Catalyst | Spark SQL logical query optimizer |
| Physical execution engine | Runs the selected physical operators; implementation evolves by Spark version |

## 2.5 Inspect the session

```python
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("SparkUI_Lab")
    .getOrCreate()
)
sc = spark.sparkContext

print("Application name:", sc.appName)
print("Application ID:", sc.applicationId)
print("Spark version:", spark.version)
print("Master:", sc.master)
print("UI enabled:", sc.getConf().get("spark.ui.enabled", "true"))
```

In most notebooks, use the existing `spark` session instead of creating multiple independent Spark contexts.

# 3. What Happens When Spark Code Runs?

```python
df = spark.range(0, 1_000_000)
df2 = df.filter("id % 2 = 0")
df3 = df2.selectExpr("id", "id * 10 AS scaled_id")
```

These operations usually build a plan; they do not necessarily process all rows immediately. Then:

```python
df3.count()
```

A simplified execution flow:
1. **Initialization:** driver and Spark execution context are available.
2. **Plan construction:** DataFrame operations form a logical plan.
3. **Analysis and optimization:** Spark resolves columns/types and optimizes the plan.
4. **Physical planning:** Spark chooses physical operators and strategies.
5. **Action:** `count()` requests a result and triggers execution if needed.
6. **Scheduling:** Spark divides work into jobs, stages, and tasks.
7. **Execution:** executors process partitions and exchange data where required.
8. **Results and metrics:** the caller receives the result; the UI records execution details.

RDDs, streaming, caching, adaptive execution, and platform-specific features add details to this simplified model.

# 4. Lazy Evaluation, Transformations, and Actions

## Transformations

Transformations describe a new dataset or plan. Examples:

```python
df.filter("id > 100")
df.select("id")
df.withColumn("double_id", F.col("id") * 2)
df.groupBy("city").count()
```

Many transformations are lazy: Spark builds a plan and executes it only when needed.

## Actions

Actions request a result or output and commonly trigger execution:

```python
df.count()
df.show()
df.take(10)
df.collect()
df.write.mode("overwrite").parquet("/some/path")
```

One action can trigger one or more jobs; a notebook cell is not the same thing as a Spark job. Avoid `collect()` on large datasets because it transfers all returned rows to the driver.

## Demonstrate it

```python
df = spark.range(0, 1_000_000)
filtered = df.filter("id > 500000")
selected = filtered.selectExpr("id", "id * 2 AS doubled")

selected.explain("formatted")  # Displays a plan; normally not a full scan
selected.count()                # Requests a result and triggers execution
```

# 5. Jobs, Stages, Tasks, Partitions, and DAGs

## Job

A **job** is execution work triggered by an action or execution request. Jobs tab entries commonly include IDs, descriptions, durations, and associated stages.

## Stage

A **stage** is a group of tasks that can be scheduled together without crossing a shuffle dependency boundary. A job may contain multiple stages. Operations such as `groupBy`, `distinct`, many joins, `repartition`, and some window operations often require shuffles.

## Task

A **task** is a unit of scheduled work, typically processing one partition of a stage. A stage with eight partitions may commonly show eight tasks, though exchanges, adaptive execution, and other operators can change task counts.

## Partition

A partition is a chunk of a distributed dataset.

```python
df = spark.range(0, 1000, numPartitions=4)
print("Partitions:", df.rdd.getNumPartitions())
print("Rows per partition:", df.rdd.glom().map(len).collect())
```

This is safe because the example collects only four small partition-size values, not all source rows.

## DAG and shuffle

```text
Source partitions
       |
       v
Filter / Select (often narrow dependencies)
       |
       v
Shuffle boundary (e.g. groupBy key)
       |
       v
Aggregate tasks
       |
       v
Result
```

A shuffle redistributes data between partitions. Narrow transformations can often process each input partition independently; shuffle dependencies commonly create stage boundaries.

**Why counts differ:** one cell may trigger multiple jobs; one job can have several stages; each stage has tasks; a task processes a partition rather than one row. Caching and reused exchanges can alter later work.

# 6. Spark UI in Local Mode

## 6.1 Default URL and port

The first active local Spark application's UI commonly uses port **4040**:

```text
http://localhost:4040
```

If the port is occupied, Spark may select a later port such as 4041. Startup logs show the actual bound address; the configured port is only the starting point.

## 6.2 Start a local session

```python
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Local_SparkUI_Demo")
    .config("spark.ui.enabled", "true")
    .getOrCreate()
)
sc = spark.sparkContext
print("Master:", sc.master)
print("Application ID:", sc.applicationId)
print("Spark version:", spark.version)
```

- `local[*]` asks Spark to use available logical CPU cores as local worker threads.
- `spark.ui.enabled=true` enables the UI and is commonly the default.
- If a session already exists, `getOrCreate()` may reuse it; builder settings may not replace settings on an already-started context.

## 6.3 Generate visible work

```python
df = spark.range(0, 5_000_000, numPartitions=8)
result = (
    df.filter(F.col("id") % 3 == 0)
      .withColumn("bucket", F.col("id") % 100)
      .groupBy("bucket")
      .count()
)
result.show()
```

The `groupBy` may require a shuffle, giving you stages and metrics to inspect.

## 6.4 Open the UI

1. Keep the Spark application running.
2. Open a browser on the same machine.
3. Visit `http://localhost:4040`.
4. If it fails, check Spark startup logs for the actual UI port.
5. Check whether the UI is disabled or another application is using the default port.

Inspect configuration:

```python
print("UI enabled:", sc.getConf().get("spark.ui.enabled", "true"))
print("Configured starting port:", sc.getConf().get("spark.ui.port", "4040"))
print("Application ID:", sc.applicationId)
```

The configuration value may still say 4040 even if Spark bound to another available port; use the logs to find the actual port.

## 6.5 Local troubleshooting

- Confirm Spark started successfully and the driver is still running.
- Check logs for the bound UI address.
- Verify `spark.ui.enabled` is not false.
- In Docker, WSL, or remote Jupyter, `localhost` may refer to a container or remote host rather than your desktop.
- Use approved port forwarding for remote environments.

# 7. Spark UI in Google Colab

Colab notebooks run in a remote runtime. Your computer's `localhost:4040` does not automatically refer to the Colab VM, so a supported port proxy or another controlled access method is needed.

## 7.1 Install and check Java

```python
!pip -q install pyspark
!java -version
```

If Java is unavailable or incompatible, install/configure a Java runtime supported by your PySpark/Spark version. Colab's base runtime changes, so do not assume one installation command works indefinitely.

## 7.2 Start Spark

```python
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Colab_SparkUI_Demo")
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
print("Spark version:", spark.version)
print("Application ID:", spark.sparkContext.applicationId)
print("Master:", spark.sparkContext.master)
```

## 7.3 Trigger a job

```python
df = spark.range(0, 2_000_000, numPartitions=8)
summary = (
    df.filter("id % 2 = 0")
      .withColumn("group_id", F.col("id") % 50)
      .groupBy("group_id")
      .count()
)
summary.show()
```

## 7.4 Accessing the UI

1. Confirm Spark is running and listening on port 4040 (check logs).
2. Use a port-proxy mechanism supported by the current Colab runtime, if available.
3. Open the returned proxy URL and keep the runtime alive.

Colab helper names and proxy URL formats may change. Use the current Colab-supported interface rather than relying on a helper copied from an old notebook. If you run Spark on a remote VM you control, an SSH tunnel is another option:

```bash
ssh -L 4040:localhost:4040 username@your-server
```

That command is for a server you can SSH into; it is not a direct Colab-only solution unless you have SSH access to the host. After tunneling, open `http://localhost:4040` locally.

## 7.5 Colab troubleshooting and security

- Check UI settings and logs for the actual port.
- The live UI generally stops being served when the driver/application ends.
- If port 4040 is occupied, configure a different starting port before SparkContext creation.
- A proxy is a transport mechanism, not a Spark feature.
- Do not expose the Spark UI publicly without access controls; it can reveal application metadata, configuration, and storage paths.

# 8. Spark UI in Databricks

Databricks is a managed platform. Normally, you do **not** navigate to `localhost:4040` yourself. The workspace exposes Spark UI and query details through compute and notebook interfaces.

## 8.1 General navigation

1. Open the Databricks workspace.
2. Open the compute resource attached to your notebook.
3. Confirm compute is running and the notebook is attached.
4. Open compute details and look for the **Spark UI** link/tab or platform Spark UI interface.
5. For individual notebook commands, inspect available command/query details and execution metrics.

Exact navigation changes by workspace version, permissions, compute type, and runtime; use the current UI labels in your workspace.

## 8.2 Generate a workload

```python
from pyspark.sql import functions as F

df = spark.range(0, 5_000_000, numPartitions=8)
result = (
    df.filter(F.col("id") % 4 == 0)
      .withColumn("bucket", F.col("id") % 100)
      .groupBy("bucket")
      .count()
)
display(result)
```

## 8.3 Places to inspect

Depending on workspace and runtime, look for:
- **Jobs:** application jobs and stage progress.
- **Stages:** task distributions, duration, shuffle, and spill metrics.
- **Executors:** task counts, memory, failures, and resource metrics.
- **SQL / Queries:** DataFrame/SQL plans and operator metrics.
- **Environment:** Spark version and configuration.
- **Driver/executor logs:** exceptions and runtime warnings.
- **Notebook query profile/details:** available in some product surfaces; names vary.

Databricks can adapt or extend the upstream Spark UI. Not every workspace displays every upstream tab in exactly the same way.

## 8.4 Notebook command vs Spark job

A notebook command timeline is not the same as a Spark job timeline. One command can trigger multiple jobs, and a DataFrame operation may appear in Spark SQL UI because DataFrame APIs use Spark SQL planning. Databricks SQL warehouse queries can have a different execution context from notebooks attached to Spark compute.

## 8.5 Troubleshooting

- Verify the notebook is attached to the intended compute.
- Confirm compute is running and you have permission to inspect its details.
- Inspect command/query details as well as Spark UI; they can show different levels of detail.
- For completed applications, use available history/query views rather than expecting a live endpoint to remain available.

# 9. Spark UI Tabs

The upstream Spark UI commonly has these tabs, though availability varies by application type and platform.

## 9.1 Jobs

Shows jobs submitted to the application, often with job ID, description, submission/completion time, duration, stage links, and progress.

**Workflow:** trigger one action → open Jobs → choose the newest job → open it → follow stage links.

A job ID is not a notebook cell number or DataFrame variable name.

## 9.2 Stages

Shows stage-level task summaries and metrics:
- Number of tasks and completed/running/failed/skipped tasks.
- Duration and task duration distribution.
- Input/output bytes.
- Shuffle read/write.
- Memory and disk spill.
- Failed task details.

Interpretation examples:
- High shuffle read/write can mean substantial data redistribution.
- Spill to disk suggests intermediate data did not fit comfortably in memory.
- A few tasks far slower than the median may indicate skew, uneven partitions, slow storage, or resource contention.
- Many tiny tasks may create scheduler overhead.
- Too few tasks for a large input may limit parallelism.

## 9.3 Storage

Shows persisted/cached RDDs and DataFrames where applicable, including partition count, memory/disk size, and cached partition fraction.

```python
cached_df = spark.range(0, 1_000_000).cache()
cached_df.count()  # Commonly materializes cached partitions
```

A Python variable or temporary view does not automatically cache data. Cache only when reuse justifies the memory/disk cost.

## 9.4 Environment

Shows Spark properties and runtime details, including many explicitly configured properties. Use it to inspect Spark version and effective application properties. Not every default appears; avoid placing secrets in Spark configuration because authorized UI users may be able to view them.

## 9.5 Executors

May show executor ID/host, active or failed status, cores, task counts, input/shuffle bytes, storage memory, GC time, failed tasks, and log links.

- **High GC time:** memory pressure or object-allocation overhead may be involved.
- **One executor does much more work:** possible skew, uneven task assignment, or resource differences.
- **Executor lost:** inspect executor/driver logs and platform events.
- **High spill:** investigate data volume, partitioning, and join/aggregation strategy.

## 9.6 SQL / DataFrame

Shows SQL/DataFrame executions, physical-plan graphs, and operator metrics when supported. Look for scans, filters, projections, exchanges, joins, aggregates, and window operators.

| Operator / term | High-level meaning |
|---|---|
| `Scan` | Reads from a source |
| `Filter` | Keeps rows satisfying a condition |
| `Project` | Produces selected or calculated columns |
| `Exchange` | Redistributes data, often a shuffle boundary |
| `HashAggregate` | Hash-based aggregation |
| `SortMergeJoin` | Sorts inputs by key and merges them |
| `BroadcastHashJoin` | Broadcasts one side and builds a local hash structure |
| `Window` | Evaluates window expressions |
| `AdaptiveSparkPlan` | Plan may be adjusted at runtime by adaptive query execution |

Exact operator names depend on Spark version and query.

## 9.7 Streaming

Structured Streaming applications may expose a Streaming tab with query/batch progress, input and processing rates, and batch durations. It appears only when relevant streaming features are used.

# 10. Reading a Job End to End

Run this lab while the UI is accessible:

```python
from pyspark.sql import functions as F

sales = (
    spark.range(0, 2_000_000, numPartitions=8)
    .withColumn("customer_id", F.col("id") % 100_000)
    .withColumn("city_id", F.col("id") % 10)
    .withColumn("amount", (F.col("id") % 5000 + 1).cast("double"))
)

result = (
    sales.filter(F.col("amount") > 1000)
         .groupBy("city_id")
         .agg(F.count("*").alias("row_count"), F.sum("amount").alias("total_amount"))
)
result.explain("formatted")
result.show()
```

Investigate:
1. Which job did the action trigger?
2. How many stages are present?
3. Which stage reads/writes shuffle data?
4. How many tasks ran in each stage?
5. Compare median and maximum task durations.
6. Are there spill metrics or failed tasks?
7. What aggregate and exchange operators appear in the SQL plan?

Record conclusions using:

```text
Observation:
Evidence from UI:
Likely explanation:
Next experiment:
```

Do not conclude that a shuffle is bad merely because it exists. Grouping and many joins require redistribution; evaluate its size and cost.

# 11. Query Plans and Spark SQL Internals

## 11.1 Logical and physical plans

- **Parsed/unresolved logical plan:** the initial expression before full name/type resolution.
- **Analyzed logical plan:** columns and types are resolved.
- **Optimized logical plan:** logical optimization rules have been applied.
- **Physical plan:** Spark chooses physical operators and strategies.
- **Adaptive/runtime plan:** in supported configurations, Spark may change parts of the plan using runtime statistics.

```python
df = spark.range(0, 1000).filter("id > 100").selectExpr("id", "id * 2 AS doubled")
df.explain()
df.explain("extended")
df.explain("formatted")
```

`explain()` describes the plan; it does not necessarily process the entire dataset. The SQL tab can show runtime metrics after execution.

## 11.2 Catalyst optimizer

Catalyst is Spark SQL's query optimizer. Depending on the query, source, and version, optimizations can include predicate pushdown, projection pruning, constant folding, expression simplification, supported join planning, and use of statistics.

## 11.3 Whole-stage code generation

For supported operators, Spark SQL can combine work into generated code to reduce per-row overhead. Whether this is used or visible depends on operators and configuration.

## 11.4 Adaptive Query Execution (AQE)

When enabled, AQE can adjust parts of a query plan using runtime statistics. Depending on Spark version and query, it can coalesce shuffle partitions, change certain join strategies, or handle skewed shuffle partitions.

```python
print(spark.conf.get("spark.sql.adaptive.enabled", "not-set"))
```

Do not assume AQE is enabled in every environment or that it resolves every skew problem.

# 12. Performance Diagnosis with the UI

## 12.1 Data skew

**Symptoms:** most tasks finish quickly but a few are much slower; maximum task duration is far above the median; a few tasks process far more data.

**Possible causes:** hot keys, uneven input partitions, or layout/partitioning mismatch.

**Experiments:** inspect key frequencies; filter/project before expensive operations where logically valid; evaluate skew-aware strategies, repartitioning, or salting only when justified; verify AQE and inspect the plan.

## 12.2 Excessive shuffle

**Symptoms:** large shuffle metrics, `Exchange` operators, and long stages after joins or aggregations.

**Experiments:** reduce rows/columns before the shuffle; check key types and duplicate-key cardinality; consider broadcast only for a genuinely small side; avoid unnecessary repartitioning. Do not remove a required shuffle at the cost of correctness.

## 12.3 Memory pressure and spill

**Symptoms:** memory/disk spill, high GC time, executor loss, or out-of-memory errors.

**Experiments:** check unnecessary caching; reduce data before expensive operators; review partition sizes and join/aggregation strategy; consult platform owners about executor memory. Do not blindly increase memory without understanding the plan and data volume.

## 12.4 Too many small tasks

**Symptoms:** very many tasks with tiny inputs and scheduler overhead large relative to useful processing.

**Experiments:** review source file sizes and partition count; consider `coalesce()` or `repartition()` only after measurement.

## 12.5 Slow input source

**Symptoms:** scan stages take a long time, possibly with low CPU utilization.

**Possible causes:** storage latency/throttling, many small files, missing partition pruning, inefficient format, or network bottlenecks.

**Experiments:** inspect scan metrics and file layout; filter on partition columns when possible; use appropriate columnar formats and file compaction where operationally justified.

## 12.6 Task/executor failures

Open failed stage/task details, then inspect driver/executor logs. The UI may show an exception summary while the full stack trace exists only in logs.

# 13. Event Logs and Spark History Server

## Live UI vs History Server

- **Live Spark UI:** served by an active application, often on a port such as 4040.
- **Event log:** record of Spark application events written when event logging is enabled.
- **Spark History Server:** separate service that reads event logs and reconstructs UI pages for completed applications.

The live UI normally stops when the application ends. The History Server is not automatically available just because Spark is installed.

## Event logging configuration

```python
spark = (
    SparkSession.builder
    .appName("HistoryServer_Demo")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///path/to/spark-events")
    .getOrCreate()
)
```

The directory must be configured and accessible to the relevant processes. In a cluster, a driver's local path may not be accessible to the History Server; shared storage may be required. Set startup properties before SparkContext creation; an existing session may not adopt them.

An operator generally must enable event logging, configure readable event-log storage, start/configure the History Server, and open its UI. Commands and paths depend on the installation. Databricks provides managed history/execution views, but retention and access depend on workspace and compute configuration; do not assume you can operate the upstream History Server on managed compute.

# 14. Useful Configuration Properties

| Property | Purpose | Notes |
|---|---|---|
| `spark.ui.enabled` | Enables/disables live UI | Commonly true by default |
| `spark.ui.port` | Starting port for UI | Common default 4040; Spark may select another available port |
| `spark.eventLog.enabled` | Enables event logging | Used for History Server reconstruction |
| `spark.eventLog.dir` | Event-log location | Must be accessible to relevant processes |
| `spark.sql.shuffle.partitions` | Default shuffle partition count for SQL operations | Tune after measuring; defaults vary by version/configuration |
| `spark.sql.adaptive.enabled` | Enables AQE | Inspect the effective setting |
| `spark.executor.memory` | Executor heap memory | Often controlled by deployment/platform |
| `spark.executor.cores` | Executor cores in supported cluster setups | Meaning differs across deployment modes |
| `spark.driver.memory` | Driver heap memory | Often must be set before driver startup |

Inspect settings:

```python
for key in [
    "spark.ui.enabled",
    "spark.ui.port",
    "spark.eventLog.enabled",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
]:
    print(key, "=", spark.conf.get(key, "not-set"))
```

Some properties are SparkContext-level rather than SQL runtime settings. `spark.conf.get()` may not expose every property in every version; `spark.sparkContext.getConf().get(...)` can help inspect application configuration.

# 15. Practical Investigation Labs

## Lab A — Narrow transformations

```python
base = spark.range(0, 1_000_000, numPartitions=8)
narrow_result = base.filter("id < 500000").selectExpr("id", "id + 1 AS next_id")
narrow_result.count()
```

Questions: How many tasks appear? Is there an exchange? Which operations can run independently per input partition?

## Lab B — Aggregation and shuffle

```python
aggregate_result = (
    base.withColumn("key", F.col("id") % 100)
        .groupBy("key")
        .agg(F.count("*").alias("rows"), F.sum("id").alias("id_sum"))
)
aggregate_result.show()
```

Questions: How many stages? Which stage reads/writes shuffle data? What is the difference between partial and final aggregation? How many output rows should there be?

## Lab C — Regular vs broadcast join

```python
facts = spark.range(0, 1_000_000).withColumn("key", F.col("id") % 1000)
dimension = (
    spark.range(0, 1000).withColumnRenamed("id", "key")
    .withColumn("label", F.concat(F.lit("K"), F.col("key")))
)
regular = facts.join(dimension, "key", "inner")
broadcasted = facts.join(F.broadcast(dimension), "key", "inner")
regular.explain("formatted")
broadcasted.explain("formatted")
regular.count()
broadcasted.count()
```

Questions: Which join strategies appear? Is there a shuffle on the large side? How does the plan change? What would you measure on representative data?

## Lab D — Cache and Storage tab

```python
cached = spark.range(0, 1_000_000).cache()
cached.count()
cached.count()
```

Questions: Does Storage show cached data? How many partitions are cached? What changes on the second action? When is caching wasteful?

# 16. Common Mistakes and Troubleshooting

1. **Assuming a cell equals a job:** one cell can trigger no job, one job, or multiple jobs.
2. **Looking for live UI after the app ends:** use event logs/History Server or platform history features.
3. **Opening the wrong `localhost`:** in Colab, Docker, WSL, or remote notebooks it may refer to the remote runtime/container.
4. **Assuming a variable materializes data:** a DataFrame variable commonly represents a plan; an action requests results.
5. **Calling every shuffle bad:** many correct aggregations and joins require shuffles.
6. **Comparing unlike workloads:** use equivalent logic, data, settings, and awareness of caching/AQE.
7. **Collecting huge data:** use `show()`, `take()`, aggregate metrics, and UI inspection instead of collecting everything.
8. **Expecting identical tabs on every platform:** managed platforms can adapt the upstream UI.
9. **Changing startup-only properties too late:** some settings must be configured before the SparkContext/driver starts.
10. **Exposing UI publicly:** protect access; the UI can reveal metadata, configuration, and paths.

# 17. Practice Assignments

## Beginner
1. Explain Spark, SparkSession, SparkContext, driver, executor, and partition in your own words.
2. Run `spark.range(0, 1000).count()` and locate the job.
3. Identify the job's stages and tasks.
4. Explain why `df.filter(...)` alone may not trigger a job.
5. Print the application ID, Spark version, and master.

## Intermediate
6. Compare UI activity for filter/select versus `groupBy()`.
7. Explain the `Exchange` operator.
8. Compare a DataFrame's partition count with task counts.
9. Cache a DataFrame, trigger an action, and inspect Storage.
10. Find an executor metric and explain its meaning.

## Advanced
11. Compare regular and broadcast joins using `explain("formatted")` and the SQL tab.
12. Use task-duration metrics to investigate a skewed dataset.
13. Describe the requirements for event logging and a History Server in local Spark.
14. Explain how AQE may adjust a plan using runtime statistics.
15. Write a diagnosis with observation, UI evidence, likely cause, and next experiment.

# 18. Quick Reference Glossary

| Term | Meaning |
|---|---|
| Spark application | Driver-coordinated workload and its executors |
| Spark UI | Web interface for application execution details |
| Driver | Coordinates execution |
| Executor | Runs tasks and can cache data |
| Cluster manager | Allocates resources in cluster deployments |
| SparkSession | Main DataFrame/SQL entry point |
| SparkContext | Core application context |
| Transformation | Operation describing a new dataset/plan |
| Action | Operation requesting results or output |
| Lazy evaluation | Work is planned and often delayed until needed |
| Job | Execution work triggered by an action/request |
| Stage | Group of tasks separated by dependencies, commonly shuffle boundaries |
| Task | Work unit typically processing a partition |
| Partition | Chunk of data processed in parallel |
| DAG | Directed acyclic graph of dependencies |
| Shuffle | Redistribution of data between partitions |
| Exchange | Physical-plan operator often representing redistribution |
| Catalyst | Spark SQL query optimizer |
| AQE | Adaptive Query Execution; runtime plan adjustment |
| Spill | Intermediate data written to disk due to memory constraints |
| Data skew | Uneven distribution of data/work across keys or partitions |
| Event log | Record of Spark application events |
| History Server | Reconstructs UI views from event logs |
| Broadcast join | Strategy that sends a small relation to executors |

## Final Takeaway

- Understand driver, executors, partitions, jobs, stages, and tasks before optimizing.
- Use Spark UI to connect code to execution metrics.
- Local mode commonly uses `http://localhost:4040`; check logs if the port differs.
- Colab runs remotely, so use a currently supported port proxy or controlled remote-access method.
- Databricks exposes Spark UI and query details through workspace/compute interfaces rather than direct `localhost:4040` access.
- Use `explain()` and the SQL tab to connect DataFrame code to physical operators.
- Base performance conclusions on task durations, shuffle, spill, GC, and query-plan evidence.
